# ResNet18 Emotion Fine-Tuning

This notebook is designed for Google Colab. It fine-tunes a pretrained `ResNet18` on a facial emotion dataset arranged in class-folder format.

Recommended dataset choice for this notebook:

- Primary dataset: `RAF-DB`
- Why: it gives a stronger real-world expression distribution than very simple starter datasets, while still being much more practical for Colab than extremely large alternatives
- Optional later upgrade: `AffectNet` if you have access, storage, and more runtime budget

For the best quality-to-practicality balance on Colab, this notebook should be used with a preprocessed RAF-DB dataset exported into `train/` and `val/` class folders.

Expected dataset structure:

```text
dataset/
  train/
    angry/
    disgusted/
    fearful/
    happy/
    neutral/
    sad/
    surprised/
  val/
    angry/
    disgusted/
    fearful/
    happy/
    neutral/
    sad/
    surprised/
```

## 2. Dataset choice

This notebook is intentionally optimized for **RAF-DB** as the default dataset.

Why RAF-DB is the best fit here:

- richer and more realistic than very small starter datasets
- strong quality for facial expression recognition
- practical enough to run in Google Colab without turning the workflow into a storage and runtime struggle

Use AffectNet only if you already have access and you are comfortable with a heavier Colab workflow.


## 3. Runtime

In Colab, set `Runtime -> Change runtime type -> GPU` before training.

## 4. Download RAF-DB into Google Drive

This notebook supports downloading a Kaggle RAF-DB mirror directly into Google Drive.

What you need before running the download cells:

1. Create a Kaggle account
2. Go to `Kaggle -> Account -> API -> Create New Token`
3. Upload the downloaded `kaggle.json` file when prompted below

Recommended Kaggle source used by this notebook:

- `shuvoalok/raf-db-dataset`

If you already have RAF-DB in Drive, you can skip the download cells and point `RAW_RAFDB_ROOT` to your existing folder.


In [ ]:
!pip install -q torch torchvision scikit-learn matplotlib seaborn PyYAML kaggle pandas

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

In [ ]:
# Upload your Kaggle API token here when running in Colab.
# Skip this cell if you already set up ~/.kaggle/kaggle.json in the current runtime.
uploaded = files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!ls -la ~/.kaggle

In [ ]:
import copy
import json
import os
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())

In [ ]:
# Update these paths before running training.
# Default recommendation: use RAF-DB as the source dataset.
# If you want the notebook to download the dataset first, keep DOWNLOAD_RAFDB = True.
# If you already have the raw RAF-DB files, set DOWNLOAD_RAFDB = False and
# point RAW_RAFDB_ROOT to the extracted RAF-DB folder in Drive.
DATASET_NAME = 'RAF-DB'
DATASET_ROOT = '/content/drive/MyDrive/rafdb_7class'
DOWNLOAD_RAFDB = True
KAGGLE_DATASET_SLUG = 'shuvoalok/raf-db-dataset'
DOWNLOAD_DIR = '/content/drive/MyDrive/RAFDB_download'
RAW_RAFDB_ROOT = '/content/drive/MyDrive/RAFDB_download/RAF-DB'
PREPROCESS_RAFDB = True
OUTPUT_DIR = '/content/drive/MyDrive/emotion_vision_outputs'
MODEL_NAME = 'resnet18'
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 12
LEARNING_RATE = 1e-4
DROPOUT = 0.2
FREEZE_BACKBONE = False

download_dir = Path(DOWNLOAD_DIR)
dataset_root = Path(DATASET_ROOT)
train_dir = dataset_root / 'train'
val_dir = dataset_root / 'val'
raw_rafdb_root = Path(RAW_RAFDB_ROOT)
output_dir = Path(OUTPUT_DIR)
download_dir.mkdir(parents=True, exist_ok=True)
dataset_root.mkdir(parents=True, exist_ok=True)
train_dir.mkdir(parents=True, exist_ok=True)
val_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('dataset:', DATASET_NAME)
print('device:', device)
print('download_dir:', download_dir)
print('raw_rafdb_root:', raw_rafdb_root)
print('dataset_root:', dataset_root)
print('train_dir exists:', train_dir.exists())
print('val_dir exists:', val_dir.exists())

## 5. Download and extract RAF-DB

This cell downloads the Kaggle RAF-DB mirror directly into Google Drive and extracts it there.

If `DOWNLOAD_RAFDB = False`, the notebook will skip this step.


In [ ]:
if DOWNLOAD_RAFDB:
    zip_path = download_dir / 'raf-db-dataset.zip'
    if not zip_path.exists():
        !kaggle datasets download -d {KAGGLE_DATASET_SLUG} -p {DOWNLOAD_DIR}
    else:
        print(f'Using existing zip: {zip_path}')

    extract_target = download_dir / 'RAF-DB'
    if not extract_target.exists():
        import zipfile
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(extract_target)
    else:
        print(f'Using existing extracted folder: {extract_target}')

    print('Top-level extracted contents:')
    for p in sorted(extract_target.iterdir()):
        print('-', p.name)
else:
    print('Skipping RAF-DB download and extraction')


## 6. Resolve the actual raw RAF-DB root

Some Kaggle archives add an extra nested folder. This cell tries to detect the folder that actually contains either:

- `DATASET/train` and `DATASET/test`, or
- `EmoLabel/list_patition_label.txt`, or
- `train_labels.csv` and `test_labels.csv`

If it finds a better nested location, it updates `raw_rafdb_root` automatically.


In [ ]:
dataset_candidates = [p.parent for p in download_dir.rglob('DATASET') if p.is_dir()] if DOWNLOAD_RAFDB else []
text_label_candidates = list(download_dir.rglob('list_patition_label.txt')) if DOWNLOAD_RAFDB else []
csv_label_candidates = list(download_dir.rglob('train_labels.csv')) if DOWNLOAD_RAFDB else []
if dataset_candidates:
    raw_rafdb_root = dataset_candidates[0]
    print('Detected RAF-DB root from DATASET folder:', raw_rafdb_root)
elif text_label_candidates:
    detected_label = text_label_candidates[0]
    raw_rafdb_root = detected_label.parent.parent
    print('Detected RAF-DB root from text labels:', raw_rafdb_root)
elif csv_label_candidates:
    detected_csv = csv_label_candidates[0]
    raw_rafdb_root = detected_csv.parent
    print('Detected RAF-DB root from csv labels:', raw_rafdb_root)
else:
    print('No downloaded label file detected automatically.')
    print('Using configured RAW_RAFDB_ROOT:', raw_rafdb_root)

print('dataset train exists:', (raw_rafdb_root / 'DATASET' / 'train').exists())
print('dataset test exists:', (raw_rafdb_root / 'DATASET' / 'test').exists())
print('text label exists:', (raw_rafdb_root / 'EmoLabel' / 'list_patition_label.txt').exists())
print('train csv exists:', (raw_rafdb_root / 'train_labels.csv').exists())
print('test csv exists:', (raw_rafdb_root / 'test_labels.csv').exists())

## 7. RAF-DB preprocessing

This cell converts raw RAF-DB files into the `train/` and `val/` class-folder layout expected by `ImageFolder`.

It supports either of these raw formats:

- Kaggle folder layout: `DATASET/train/1..7` and `DATASET/test/1..7`
- official RAF-DB text labels: `EmoLabel/list_patition_label.txt`
- Kaggle mirror CSV labels: `train_labels.csv` and `test_labels.csv`

If you already have a prepared folder at `DATASET_ROOT`, set `PREPROCESS_RAFDB = False` and skip this step.


In [ ]:
RAFDB_LABEL_MAP = {
    1: 'surprised',
    2: 'fearful',
    3: 'disgusted',
    4: 'happy',
    5: 'sad',
    6: 'angry',
    7: 'neutral',
}

def resolve_rafdb_image(raw_root: Path, original_name: str) -> Path:
    stem = Path(original_name).stem
    candidates = [
        raw_root / 'Image' / 'aligned' / f'{stem}_aligned.jpg',
        raw_root / 'Image' / 'aligned' / original_name,
        raw_root / 'Image' / f'{stem}_aligned.jpg',
        raw_root / 'Image' / original_name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Could not resolve RAF-DB image for {original_name}')

def preprocess_rafdb_text(raw_root: Path, target_root: Path) -> None:
    label_file = raw_root / 'EmoLabel' / 'list_patition_label.txt'
    if not label_file.exists():
        raise FileNotFoundError(f'Missing label file: {label_file}')

    for split in ['train', 'val']:
        for class_name in RAFDB_LABEL_MAP.values():
            (target_root / split / class_name).mkdir(parents=True, exist_ok=True)

    copied = {'train': 0, 'val': 0}
    with label_file.open('r') as handle:
        for line in handle:
            filename, label_str = line.strip().split()
            label = int(label_str)
            class_name = RAFDB_LABEL_MAP[label]
            split = 'train' if filename.startswith('train') else 'val'
            source = resolve_rafdb_image(raw_root, filename)
            destination = target_root / split / class_name / source.name
            if not destination.exists():
                shutil.copy2(source, destination)
                copied[split] += 1

    print('RAF-DB text preprocessing complete')
    print(copied)

def preprocess_rafdb_folder_layout(raw_root: Path, target_root: Path) -> None:
    source_train = raw_root / 'DATASET' / 'train'
    source_test = raw_root / 'DATASET' / 'test'
    if not source_train.exists() or not source_test.exists():
        raise FileNotFoundError(
            f'Missing Kaggle folder layout. Expected {source_train} and {source_test}'
        )

    copied = {'train': 0, 'val': 0}
    split_map = {'train': source_train, 'val': source_test}

    for split_name, split_dir in split_map.items():
        for label_id, class_name in RAFDB_LABEL_MAP.items():
            src_class_dir = split_dir / str(label_id)
            dst_class_dir = target_root / split_name / class_name
            dst_class_dir.mkdir(parents=True, exist_ok=True)

            if not src_class_dir.exists():
                print(f'Skipping missing source directory: {src_class_dir}')
                continue

            for image_path in src_class_dir.rglob('*'):
                if not image_path.is_file():
                    continue
                destination = dst_class_dir / image_path.name
                if not destination.exists():
                    shutil.copy2(image_path, destination)
                    copied[split_name] += 1

    print('RAF-DB folder-layout preprocessing complete')
    print(copied)

def find_image_file(images_root: Path, image_name: str) -> Path:
    direct = images_root / image_name
    if direct.exists():
        return direct

    stem = Path(image_name).stem
    candidates = list(images_root.rglob(f'{stem}*'))
    if candidates:
        return candidates[0]

    raise FileNotFoundError(f'Could not find image for {image_name}')

def preprocess_rafdb_csv(raw_root: Path, target_root: Path) -> None:
    train_csv = raw_root / 'train_labels.csv'
    test_csv = raw_root / 'test_labels.csv'

    if not train_csv.exists() or not test_csv.exists():
        raise FileNotFoundError(
            f'Missing CSV labels. Expected {train_csv} and {test_csv}'
        )

    image_dirs = [p for p in raw_root.iterdir() if p.is_dir()]
    if not image_dirs:
        raise FileNotFoundError(f'No image directories found inside {raw_root}')

    print('Available directories inside raw root:')
    for d in image_dirs:
        print('-', d.name)

    images_root = max(image_dirs, key=lambda p: len(list(p.rglob('*'))))
    print('Using images_root:', images_root)

    for split in ['train', 'val']:
        for class_name in RAFDB_LABEL_MAP.values():
            (target_root / split / class_name).mkdir(parents=True, exist_ok=True)

    split_specs = [
        ('train', train_csv),
        ('val', test_csv),
    ]

    copied = {'train': 0, 'val': 0}

    for split_name, csv_path in split_specs:
        df = pd.read_csv(csv_path)
        print(f'\nColumns in {csv_path.name}: {list(df.columns)}')

        filename_col = None
        label_col = None
        for col in df.columns:
            col_lower = col.lower()
            if filename_col is None and ('image' in col_lower or 'file' in col_lower or 'name' in col_lower):
                filename_col = col
            if label_col is None and ('label' in col_lower or 'emotion' in col_lower or 'class' in col_lower):
                label_col = col

        if filename_col is None:
            filename_col = df.columns[0]
        if label_col is None:
            label_col = df.columns[1]

        print('Using filename column:', filename_col)
        print('Using label column:', label_col)

        for _, row in df.iterrows():
            image_name = str(row[filename_col])
            label = int(row[label_col])
            class_name = RAFDB_LABEL_MAP[label]

            source = find_image_file(images_root, image_name)
            destination = target_root / split_name / class_name / source.name

            if not destination.exists():
                shutil.copy2(source, destination)
                copied[split_name] += 1

    print('\nRAF-DB CSV preprocessing complete')
    print(copied)

if PREPROCESS_RAFDB:
    if (raw_rafdb_root / 'DATASET' / 'train').exists() and (raw_rafdb_root / 'DATASET' / 'test').exists():
        preprocess_rafdb_folder_layout(raw_rafdb_root, dataset_root)
    elif (raw_rafdb_root / 'train_labels.csv').exists() and (raw_rafdb_root / 'test_labels.csv').exists():
        preprocess_rafdb_csv(raw_rafdb_root, dataset_root)
    else:
        preprocess_rafdb_text(raw_rafdb_root, dataset_root)
else:
    print('Skipping RAF-DB preprocessing')


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomRotation(8),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(val_dir, transform=val_transform)

class_names = train_dataset.classes
print('classes:', class_names)
print('train samples:', len(train_dataset))
print('val samples:', len(val_dataset))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

if FREEZE_BACKBONE:
    for param in model.parameters():
        param.requires_grad = False

in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(DROPOUT),
    nn.Linear(in_features, len(class_names)),
)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)

print(model.fc)

In [ ]:
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_labels = []
    all_preds = []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, labels)

            if is_train:
                loss.backward()
                optimizer.step()

        total_loss += float(loss.item()) * images.size(0)
        preds = logits.argmax(dim=1)
        all_labels.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())

    avg_loss = total_loss / max(len(loader.dataset), 1)
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, acc, macro_f1, all_labels, all_preds


In [ ]:
history = []
best_val_f1 = -1.0
best_state = None

for epoch in range(EPOCHS):
    train_loss, train_acc, train_f1, _, _ = run_epoch(model, train_loader, optimizer)
    val_loss, val_acc, val_f1, val_labels, val_preds = run_epoch(model, val_loader)

    row = {
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'train_f1': train_f1,
        'val_loss': val_loss,
        'val_acc': val_acc,
        'val_f1': val_f1,
    }
    history.append(row)
    print(row)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = copy.deepcopy(model.state_dict())

print('best val f1:', best_val_f1)

In [ ]:
model.load_state_dict(best_state)
val_loss, val_acc, val_f1, val_labels, val_preds = run_epoch(model, val_loader)

print('final val accuracy:', val_acc)
print('final val macro f1:', val_f1)
print(classification_report(val_labels, val_preds, target_names=class_names))

cm = confusion_matrix(val_labels, val_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Validation Confusion Matrix')
plt.show()

In [ ]:
checkpoint_path = output_dir / 'emotion_resnet18.pt'
metrics_path = output_dir / 'emotion_resnet18_metrics.json'

torch.save(best_state, checkpoint_path)
with metrics_path.open('w') as f:
    json.dump({
        'class_names': class_names,
        'best_val_f1': best_val_f1,
        'final_val_accuracy': val_acc,
        'final_val_macro_f1': val_f1,
        'history': history,
    }, f, indent=2)

print('saved checkpoint to', checkpoint_path)
print('saved metrics to', metrics_path)

## After training

Download `emotion_resnet18.pt` and place it in:

- `models/emotion/`

Then update `configs/emotion/base.yaml` if needed and switch the backend from `mock` mode to `local` mode.